# Deepfake Audio Detection Pipeline (Kaggle Ready)

This notebook contains the complete pipeline for preprocessing audio, extracting MFCC features, building a CNN model in PyTorch, training the model, and evaluating its performance.

In [ ]:
!pip install -q librosa scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import glob
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 150 # Max frames for MFCC (approx 3 seconds)
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
MAX_SAMPLES_PER_CLASS = 2000 # Subsample to avoid memory issues and speed up training

if os.path.exists('/kaggle/input'):
    # Kaggle dataset path structure
    DATA_DIR = '/kaggle/input/the-fake-or-real-dataset/for-norm/for-norm/training'
    if not os.path.exists(DATA_DIR):
        # fallback to the one in reference notebook if different
        DATA_DIR = '/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm/training'
else:
    DATA_DIR = 'dataset/LA_norm/train'
    
print(f"Using data directory: {DATA_DIR}")

In [ ]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        if mfccs.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :MAX_LEN]
        return mfccs
    except Exception as e:
        return None

class AudioDataset(Dataset):
    def __init__(self, data_list, labels):
        self.data_list = data_list
        self.labels = labels
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        features = extract_features(self.data_list[idx])
        if features is None:
            features = np.zeros((N_MFCC, MAX_LEN))
        features = features[np.newaxis, ...]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        self.flat_size = 64 * (N_MFCC // 8) * (MAX_LEN // 8)
        self.fc1 = nn.Linear(self.flat_size, 128)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(-1, self.flat_size)
        x = self.dropout(self.relu4(self.fc1(x)))
        x = self.fc2(x)
        return self.sigmoid(x).squeeze()

In [ ]:
def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.absolute((fnr - fpr)))
    return fpr[eer_index]

# Load Data
genuine_dir = os.path.join(DATA_DIR, 'real') # Using 'real' as in the reference kaggle dataset
spoof_dir = os.path.join(DATA_DIR, 'fake')   # Using 'fake'

if not os.path.exists(genuine_dir):
    genuine_dir = os.path.join(DATA_DIR, 'genuine')
    spoof_dir = os.path.join(DATA_DIR, 'spoof')

genuine_files = glob.glob(os.path.join(genuine_dir, '*.wav'))[:MAX_SAMPLES_PER_CLASS]
spoof_files = glob.glob(os.path.join(spoof_dir, '*.wav'))[:MAX_SAMPLES_PER_CLASS]

files = genuine_files + spoof_files
labels = [1] * len(genuine_files) + [0] * len(spoof_files)

print(f"Loaded {len(genuine_files)} Genuine and {len(spoof_files)} Deepfake audio files.")

In [ ]:
if len(files) > 0:
    X_train, X_test, y_train, y_test = train_test_split(files, labels, test_size=0.2, random_state=42, stratify=labels)
    train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}...")
    
    model = AudioCNN().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_f1 = 0

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        model.eval()
        all_targets, all_outputs = [], []
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
                all_targets.extend(targets.cpu().numpy())
                all_outputs.extend(outputs.cpu().numpy())
                
        predictions = (np.array(all_outputs) > 0.5).astype(int)
        acc = accuracy_score(all_targets, predictions)
        f1 = f1_score(all_targets, predictions)
        print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
        
        if f1 >= best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), 'deepfake_audio_model.pth')
else:
    print("No audio files found to train on.")

In [ ]:
if len(files) > 0:
    print("\n--- Final Performance Report ---")
    model.load_state_dict(torch.load('deepfake_audio_model.pth'))
    model.eval()
    all_targets, all_outputs = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            all_targets.extend(targets.numpy())
            all_outputs.extend(outputs.cpu().numpy())
            
    predictions = (np.array(all_outputs) > 0.5).astype(int)
    print("Confusion Matrix:\n", confusion_matrix(all_targets, predictions))
    print(f"Overall Accuracy: {accuracy_score(all_targets, predictions):.4f}")
    print(f"F1 Score: {f1_score(all_targets, predictions):.4f}")
    print(f"Equal Error Rate (EER): {compute_eer(all_targets, all_outputs):.4f}")
    print("\nTraining complete! The model 'deepfake_audio_model.pth' is saved in the working directory.")